In [ ]:
from paper2_navigation.config import INTERIM_DATA_DIR, PROCESSED_DATA_DIR, FIGURES_DIR, MODELS_DIR, EXPERIMENT_DATA_DIR, RESULTS_DIR
import os
import pickle
import numpy as np
import keras
import paper2_navigation.data.preprocessing as p
import matplotlib.pyplot as plt
import pandas as pd
from keras_tuner import HyperParameters
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, BatchNormalization, LayerNormalization, Input, Bidirectional, Attention, Add
from tensorflow.keras.models import Model
import keras_tuner as kt
import json

import paper2_navigation.modeling.gen_cnc_vlp_model as vlp

import paper2_navigation.data.experiment_processing as ep

# import paper2_navigation.data.path_geometry as pg
# import paper2_navigation.plots.model_performance_plotting as pp
import paper2_navigation.modeling.rnn as rnn
import paper2_navigation.stats.navigation_performance as nav

from tensorflow import keras
import tensorflow as tf

%load_ext autoreload
%autoreload 2

import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')


from paper2_navigation.modeling.rnn_config import GLOBAL_CONFIG

FULL_TUNING = os.getenv('PAPER2_FULL_TUNING', '0') == '1'
MAX_TRIALS = 200 if FULL_TUNING else 1
TUNING_EPOCHS = 100 if FULL_TUNING else 1
REGULARIZATION_EPOCHS = 50 if FULL_TUNING else 1
FINAL_EPOCHS = 500 if FULL_TUNING else 1
TUNING_ROOT = RESULTS_DIR / ('rnn_tuning_full' if FULL_TUNING else 'rnn_tuning_smoke')
TUNING_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG_OUTPUT = TUNING_ROOT / 'config.json'
print('Execution mode:', 'full tuning' if FULL_TUNING else 'smoke validation')

In [ ]:
vlp_models = vlp.load_vlp_models()
vlp_model = vlp_models['high_acc']
# df_lst = []
# for i in range(10):
#     path = EXPERIMENT_DATA_DIR/f'Robot/exp1_{i}.csv'
#     df_lst.append(ep.process_robot_exp_file(path, vlp_model))

df_lst = []
for i in range(10):
    test_file = INTERIM_DATA_DIR / 'navigation_training'/f'exp1_{i}_high_acc.csv'
    df = pd.read_csv(test_file)
    df_lst.append(df)

train_files = df_lst[:-2]
valid_files = df_lst[-2:-1]
test_files = df_lst[-1:]

In [ ]:
import paper2_navigation.modeling.rnn as rnn


def expand(y):
    y_rad = y[:,2] 
    y_angles = np.column_stack((np.sin(y_rad), np.cos(y_rad)))
    return [y[:,:2], y_angles]

# Define a cosine similarity–based loss function for headings
def cosine_loss(y_true, y_pred):
    # Use tf.keras.losses.CosineSimilarity which returns negative values (max similarity = -1)
    cos_sim = tf.keras.losses.CosineSimilarity(axis=1)(y_true, y_pred)
    # Convert to a loss (0 when identical, higher when misaligned)
    return 1 + cos_sim  # When vectors are identical, cos_sim = -1, so loss becomes 0


def preprocess_df(df):
    X = df[['vlp_x_hist', 'vlp_y_hist','vlp_heading_hist_rad','vlp_heading_change_rad', 'encoder_heading_change_rad', 'encoder_heading_hist_rad', 'encoder_x_hist','encoder_y_hist']].values
    y = df[['x_hist', 'y_hist','heading_hist_rad']].values
    X_win, y_win, m_win = p.window_data(X, y, y, overlap = 0.999999, window_len = 25)
    return X_win, y_win, m_win


def read_csv_to_train(file_list):
    X_lst = []
    y_lst = []
    m_lst = []

    for df in file_list:
        X_win, y_win, m_win = preprocess_df(df)
        X_lst.append(X_win)
        y_lst.append(y_win)
        m_lst.append(m_win)


    X = np.concatenate(X_lst, axis = 0)
    y = np.concatenate(y_lst, axis = 0)
    m = np.concatenate(m_lst, axis = 0)

    return X, y, m

X_train, y_train, _ = read_csv_to_train(train_files)
X_valid, y_valid, _ = read_csv_to_train(valid_files)
X_test, y_test, _ = read_csv_to_train(test_files)

X_train = np.nan_to_num(X_train, nan = 0)
X_valid = np.nan_to_num(X_valid, nan = 0)
X_test = np.nan_to_num(X_test, nan = 0)

# after you've built `X_train` with shape (N, window, 8):
flat = X_train.reshape(-1, X_train.shape[-1])
rnn.global_norm.adapt(flat)

# 2) grab its statistics and push them into your module globals
rnn.FEAT_MEAN = rnn.global_norm.mean.numpy()
rnn.FEAT_VAR  = rnn.global_norm.variance.numpy()



## RNN Hyperparameter tuning

### Stage 1 parameter tuning - neurons and layers

In [ ]:

tuner = kt.RandomSearch(
    rnn.build_architecture_model,   # ← use your config‑driven builder
    objective="val_loss",
    max_trials=MAX_TRIALS,
    directory=str(TUNING_ROOT / 'stage1'),
    project_name="rnn_random_search",
    overwrite=not FULL_TUNING
)

tensorboard_callback = tf.keras.callbacks.TensorBoard(
    str(TUNING_ROOT / 'logs_stage1'), histogram_freq=1
)

tuner.search(
    X_train,
    expand(y_train),
    epochs=TUNING_EPOCHS,
    validation_split=0.2,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True
        ),
        tensorboard_callback
    ]
)


In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]

In [ ]:
# Extract the top 10 trials from your tuner
best_trials = tuner.oracle.get_best_trials(num_trials=10)

rows = []
for trial in best_trials:
    hp = trial.hyperparameters
    # Pull out architecture hyperparameters
    num_layers = hp.get('num_layers')
    rec_units = [hp.get(f'recurrent_units_{i}') for i in range(num_layers)]
    # Pad to a fixed width (max 4 layers)
    rec_units += [None] * (4 - len(rec_units))
    # Get the best validation loss achieved by this trial
    val_loss = trial.metrics.get_best_value('val_loss')
    
    rows.append({
        'trial_id': trial.trial_id,
        'val_loss': val_loss,
        'num_layers': num_layers,
        'units_layer_0': rec_units[0],
        'units_layer_1': rec_units[1],
        'units_layer_2': rec_units[2],
        'units_layer_3': rec_units[3],
    })

# Build and display a DataFrame
df_top10 = pd.DataFrame(rows).sort_values('val_loss').reset_index(drop=True)
df_top10



# 1) Grab the best HP from stage 1
best_hp = tuner.get_best_hyperparameters(1)[0]

# 2) Update GLOBAL_CONFIG in memory
GLOBAL_CONFIG["best_architecture"] = {
    "num_layers": best_hp.get("num_layers"),
    "recurrent_units": [
        best_hp.get(f"recurrent_units_{i}") 
        for i in range(best_hp.get("num_layers"))
    ]
}

# 3) (Optional) Persist to disk as config.json
with open(CONFIG_OUTPUT, "w") as f:
    json.dump(GLOBAL_CONFIG, f, indent=2)

print("✅ GLOBAL_CONFIG updated:", GLOBAL_CONFIG["best_architecture"])

In [ ]:
df_top10

### Stage 2 tuning - regularization

In [ ]:
tuner = kt.RandomSearch(
    rnn.build_regularization_model,  # Your stage 2 model-building function
    objective="val_loss",
    max_trials=MAX_TRIALS,
    directory=str(TUNING_ROOT / 'stage2'),
    project_name="rnn_reg_tuning",
    overwrite=not FULL_TUNING
)

# ✅ Enable TensorBoard Logging
tensorboard_callback = tf.keras.callbacks.TensorBoard(str(TUNING_ROOT / 'logs_stage2'), histogram_freq=1)

# ✅ Start tuning with random search
tuner.search(
    X_train, expand(y_train),
    epochs=REGULARIZATION_EPOCHS,
    validation_split=0.2,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        tensorboard_callback
    ]
)

In [ ]:
# get the top 10 trials
best_trials = tuner.oracle.get_best_trials(num_trials=100)

rows = []
for trial in best_trials:
    hp = trial.hyperparameters
    num_layers = GLOBAL_CONFIG["best_architecture"]["num_layers"]
    # pull out each layer’s regularization HPs
    dr   = [ hp.get(f"dropout_{i}")           for i in range(num_layers) ]
    rdr  = [ hp.get(f"recurrent_dropout_{i}") for i in range(num_layers) ]
    bn   = [ hp.get(f"batch_norm_{i}")        for i in range(num_layers) ]
    ln   = [ hp.get(f"layer_norm_{i}")        for i in range(num_layers) ]
    val_loss = trial.metrics.get_best_value("val_loss")

    rows.append({
      "trial_id":       trial.trial_id,
      "val_loss":       val_loss,
      **{f"dropout_{i}":           dr[i]  for i in range(num_layers)},
      **{f"recurrent_dropout_{i}": rdr[i] for i in range(num_layers)},
      **{f"batch_norm_{i}":        bn[i]  for i in range(num_layers)},
      **{f"layer_norm_{i}":        ln[i]  for i in range(num_layers)},
    })

df_reg_top100 = pd.DataFrame(rows).sort_values("val_loss").reset_index(drop=True)
df_reg_top100


### Check no regularisation

In [ ]:
from keras_tuner.engine.hyperparameters import HyperParameters
class FixedHyperParameters(HyperParameters):
    def __init__(self):
        super().__init__()

    def Choice(self, name, values, default=None):
        return 0.0  # Always return 0.0 for dropout and recurrent_dropout

    def Boolean(self, name):
        return False  # Always return False for all normalization options

fixed_hp = FixedHyperParameters()
baseline_model = rnn.build_regularization_model(fixed_hp)

history = baseline_model.fit(
    X_train, expand(y_train),
    validation_split=0.2,
    epochs=REGULARIZATION_EPOCHS,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),

    ]
)


In [ ]:
df_reg_top100

In [ ]:
# Run this right after tuner.search(...)
best_hp = tuner.get_best_hyperparameters(1)[0]

# Build your new regularization_defaults entry
num_layers = GLOBAL_CONFIG["best_architecture"]["num_layers"]
GLOBAL_CONFIG["regularization_defaults"] = {
  "dropout": [best_hp.get(f"dropout_{i}") for i in range(num_layers)],
  "recurrent_dropout": [best_hp.get(f"recurrent_dropout_{i}") for i in range(num_layers)],
  "batch_norm": [best_hp.get(f"batch_norm_{i}") for i in range(num_layers)],
  "layer_norm": [ best_hp.get(f"layer_norm_{i}") for i in range(num_layers) ]
}
# 3) (Optional) Persist to disk as config.json
with open(CONFIG_OUTPUT, "w") as f:
    json.dump(GLOBAL_CONFIG, f, indent=2)
print("✅ GLOBAL_CONFIG updated:", GLOBAL_CONFIG["best_architecture"])

### Stage 3 tuning - Optimization parameters

In [ ]:
tuner = kt.RandomSearch(
    rnn.build_optimization_model,  # Stage 3 model function
    objective="val_loss",
    max_trials=MAX_TRIALS,
    directory=str(TUNING_ROOT / 'stage3'),
    project_name="rnn_opt_tuning",
    overwrite=not FULL_TUNING
)

tensorboard_callback = tf.keras.callbacks.TensorBoard(str(TUNING_ROOT / 'logs_stage3'), histogram_freq=1)

tuner.search(
    X_train, expand(y_train),
    epochs=TUNING_EPOCHS,
    validation_split=0.2,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        tensorboard_callback
    ]
)


In [ ]:

best_hp = tuner.get_best_hyperparameters(1)[0]

opt_cfg = {
    "lr":         float(best_hp.get("lr")),
    "scheduler":  best_hp.get("scheduler"),
    "optimizer":  best_hp.get("optimizer"),
}

if opt_cfg["scheduler"] == "cosine":
    opt_cfg["decay_steps"] = int(best_hp.get("decay_steps_cosine"))
elif opt_cfg["scheduler"] == "exponential":
    # note the full name here:
    opt_cfg["decay_steps"] = int(best_hp.get("decay_steps_exponential"))
    opt_cfg["decay_rate"]  = float(best_hp.get("decay_rate_exp"))

GLOBAL_CONFIG["optimization_defaults"] = opt_cfg
with open(CONFIG_OUTPUT, "w") as f:
    json.dump(GLOBAL_CONFIG, f, indent=2)

print("✅ NEW optimization_defaults:", GLOBAL_CONFIG["optimization_defaults"])


In [ ]:
# 1) Grab the top-10 (or fewer if you only ran 5) trials
best_trials = tuner.oracle.get_best_trials(num_trials=10)

rows = []
for trial in best_trials:
    hp       = trial.hyperparameters
    val_loss = trial.metrics.get_best_value("val_loss")

    # these are the tuned HPs in build_optimization_model:
    lr          = hp.get("lr")
    sched       = hp.get("scheduler")
    # conditionally pull out the scheduler params
    decay_cos   = hp.get("decay_steps_cosine") if sched == "cosine"      else None
    decay_exp   = hp.get("decay_steps_exponential")   if sched == "exponential" else None
    rate_exp    = hp.get("decay_rate_exp")    if sched == "exponential" else None
    optimizer   = hp.get("optimizer")

    rows.append({
        "trial_id":            trial.trial_id,
        "val_loss":            val_loss,
        "lr":                  lr,
        "scheduler":           sched,
        "decay_steps_cosine":  decay_cos,
        "decay_steps_exponential": decay_exp,
        "decay_rate_exponential":  rate_exp,
        "optimizer":           optimizer,
    })

# 2) Build & sort your DataFrame
df_opt_top10 = (
    pd.DataFrame(rows)
      .sort_values("val_loss")
      .reset_index(drop=True)
)

# 3) Inspect
print(df_opt_top10)


### Stage 4 Tuning - Other hyperparameters

In [ ]:
tuner = kt.GridSearch(
    rnn.build_stage4_model,  # Stage 4 model function
    objective=kt.Objective("val_loss", direction="min"),
    directory=str(TUNING_ROOT / 'stage4'),
    project_name="rnn_extra_tuning",
    overwrite=not FULL_TUNING
)

tensorboard_callback = tf.keras.callbacks.TensorBoard(str(TUNING_ROOT / 'logs_stage4'), histogram_freq=1)

tuner.search(
    X_train, expand(y_train),
    epochs=TUNING_EPOCHS,
    validation_split=0.2,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        tensorboard_callback
    ]
)


In [ ]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best hyperparameters:")
print(best_hps.values)

In [ ]:

# 1) After your tuner.search(...)
best_hp = tuner.get_best_hyperparameters(1)[0]

# 2) Pull out the winning window‐length choice
best_seq = int(best_hp.get("sequence_length"))

# 3) Update your in‐memory config
GLOBAL_CONFIG["sequence_length"]["use_length"] = best_seq

# 4) Persist it to disk
with open(CONFIG_OUTPUT, "w") as f:
    json.dump(GLOBAL_CONFIG, f, indent=2)

print(f"✅ GLOBAL_CONFIG.sequence_length.use_length set to {best_seq}")


In [ ]:
# collect all trials
rows = []
for trial in tuner.oracle.trials.values():
    hp       = trial.hyperparameters
    seq_len  = hp.get("sequence_length")
    # get the best val_loss this trial ever saw
    val_loss = trial.metrics.get_best_value("val_loss")
    rows.append({
        "trial_id":        trial.trial_id,
        "sequence_length": seq_len,
        "val_loss":        val_loss
    })

# build a DataFrame and sort by loss
df_stage4 = (
    pd.DataFrame(rows)
      .sort_values("val_loss")
      .reset_index(drop=True)
)

df_stage4


### Final model

In [ ]:
best_model = rnn.build_final_model()
final_sequence_length = GLOBAL_CONFIG['sequence_length']['use_length']


history = best_model.fit(
    x=X_train[:, -final_sequence_length:, :],
    y = expand(y_train),
    validation_data=(X_valid[:, -final_sequence_length:, :], expand(y_valid)),
    epochs=FINAL_EPOCHS,
    batch_size=32, 
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)]
)

In [ ]:
model_output = MODELS_DIR / 'rnn/rnn.keras' if FULL_TUNING else RESULTS_DIR / 'rnn_smoke.keras'
best_model.save(model_output)
print('Saved model:', model_output)

In [ ]:
best_model.summary()